# Cervical Cancer Classifier — Kaggle Training Notebook

## What you need before running

| Requirement | How to set it up |
|---|---|
| **Herlev dataset** | Add Data → search `shubhrawat132/herlevdataset` — already attached at `/kaggle/input/herlevdataset` |
| **Internet ON** | Notebook Settings → Internet → ON  (needed to `git clone` the repo) |
| **GPU ON** | Notebook Settings → Accelerator → GPU T4 x2 |

No second dataset upload needed — the repo is cloned from GitHub in Cell 1.

## Flow
1. Clone repo from GitHub
2. Inspect dataset structure
3. Install requirements
4. Configure hyperparameters
5. Train
6. Zip & download checkpoints

In [ ]:
# ── Cell 1 — Find or clone the repo ───────────────────────────────────────────
# Try to find the repo in:
#   1. /kaggle/input/<dataset> (if uploaded as Kaggle Dataset)
#   2. Clone from GitHub if not found
#
# Requires: Internet ON to clone from GitHub

import os
import subprocess, sys
from pathlib import Path

DEFAULT_REPO_URL = 'https://github.com/Shubh-Rawat7/Cervical-Cancer-Classifier.git'
REPO_URL = os.environ.get('REPO_URL', DEFAULT_REPO_URL).strip()
# If you pushed your refactored code to a different GitHub repository,
# set the Kaggle environment variable REPO_URL or edit this default URL.
REPO_DIR = None

print(f'Using REPO_URL = {REPO_URL}')

# ── Try to find uploaded repo in Kaggle input ─────────────────────────────────
def find_uploaded_repo() -> Path | None:
    for candidate in Path('/kaggle/input').iterdir():
        if not candidate.is_dir():
            continue
        backend_dir = candidate / 'backend'
        if (backend_dir / 'train.py').exists():
            return candidate
    return None

REPO_DIR = find_uploaded_repo()
if REPO_DIR is not None:
    print(f'Found repo in uploaded dataset: {REPO_DIR}')
else:
    REPO_DIR = Path('/kaggle/working/repo')
    if REPO_DIR.exists():
        print('Repo already cloned — pulling latest changes...')
        subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull'])
    else:
        print('Cloning repo from GitHub...')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])

BACKEND_DIR = REPO_DIR / 'backend'

# ── Validate ──────────────────────────────────────────────────────────────────
if not BACKEND_DIR.exists():
    raise FileNotFoundError(
        f'backend/ not found under {REPO_DIR}\n'
        'Options:\n'
        '  1. Upload your project as a Kaggle Dataset: Add Data → Upload Data\n'
        '  2. Set REPO_URL to a GitHub repo that contains your refactored code.\n'
        '  3. Or clone from GitHub with Internet ON and a valid repo URL.\n'
    )

if not (BACKEND_DIR / 'train.py').exists():
    repo_files = '\n'.join(
        f'  {p.relative_to(REPO_DIR)}' for p in sorted(REPO_DIR.iterdir())
    )
    raise FileNotFoundError(
        f'backend/train.py not found under {REPO_DIR}\n'
        f'Files found at top-level:\n{repo_files}\n\n'
        'The default GitHub repo does not contain the refactored backend.\n'
        'Upload your project as a Kaggle Dataset or set REPO_URL to your repo URL.\n'
    )

print(f'\nRepo      : {REPO_DIR}')
print(f'Backend   : {BACKEND_DIR}')
print('\nFiles in backend/:')
for f in sorted(BACKEND_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name}')

In [ ]:
# ── Cell 2 — Inspect the Herlev dataset ──────────────────────────────────────
# Run this to see EXACTLY what paths Kaggle mounted your dataset at.
# Copy the path that contains your class folders into DATA_DIR if Cell 3 fails.

from pathlib import Path

CLASS_NAMES = ['Normal', 'CIN1', 'CIN2', 'CIN3', 'Cancer']

print('=== /kaggle/input tree (dirs only, max depth 5) ===')
for p in sorted(Path('/kaggle/input').rglob('*')):
    depth = len(p.relative_to('/kaggle/input').parts)
    if depth > 5:
        continue
    if p.is_dir():
        # Count images if it looks like a class folder
        n_imgs = len([f for f in p.iterdir() if f.is_file()]) if p.is_dir() else 0
        has_classes = sum((p / c).exists() for c in CLASS_NAMES)
        tag = f'  ← {has_classes} class folders found, {n_imgs} files' if has_classes >= 2 else (f'  ({n_imgs} files)' if n_imgs else '')
        print(f'  {"/" * depth}{p.name}/{tag}')

print()
print('=== Looking for class folders ===')
for p in sorted(Path('/kaggle/input').rglob('*')):
    if p.is_dir() and p.name in CLASS_NAMES:
        n = len([f for f in p.iterdir() if f.is_file()])
        print(f'  {p}  ({n} images)')


In [ ]:
# ── Cell 3 — Resolve DATA_DIR and install requirements ───────────────────────
# Finds whichever folder contains the class images.
# Priority: flat layout (all classes in one folder) > pre-split train/ folder
# Never picks test/ or val/ alone — train.py does its own stratified split.

import subprocess, sys
from pathlib import Path

CLASS_NAMES = ['Normal', 'CIN1', 'CIN2', 'CIN3', 'Cancer']

def count_class_images(folder: Path, class_names) -> int:
    """Count total images directly inside class subfolders."""
    total = 0
    for c in class_names:
        d = folder / c
        if d.exists():
            total += len([f for f in d.iterdir() if f.is_file()])
    return total

def classes_present(folder: Path, class_names, min_classes: int = 2) -> bool:
    return sum((folder / c).exists() for c in class_names) >= min_classes

def find_data_dir(root: Path) -> 'Path | None':
    """
    Return the best folder to pass as --data-dir.
    Rules (in priority order):
      1. A folder that directly contains >=2 class subdirs AND is NOT named
         'test' or 'val' (flat layout — best case for --val-split training).
      2. If dataset is pre-split into train/val/test, return the 'train' folder
         so train.py sees all training images and does its own val split.
      3. Recursive search: pick the non-test/val candidate with the most images.
    """
    if not root.exists():
        return None

    # ── Rule 1: flat layout at root level ────────────────────────────────────
    if classes_present(root, CLASS_NAMES) and root.name.lower() not in ('test', 'val', 'validation'):
        return root

    # ── Rule 2: pre-split — prefer train/ folder ─────────────────────────────
    train_dir = root / 'train'
    if train_dir.exists() and classes_present(train_dir, CLASS_NAMES):
        print(f'  [find_data_dir] Dataset is pre-split. Using train/ folder: {train_dir}')
        print(f'  WARNING: val/ and test/ splits will be ignored; '
              f'train.py will create its own val split from train/ images.')
        return train_dir

    # ── Rule 3: recursive — pick biggest non-test/val candidate ──────────────
    candidates = []
    for candidate in sorted(root.rglob('*')):
        if not candidate.is_dir():
            continue
        if candidate.name.lower() in ('test', 'val', 'validation'):
            continue
        if classes_present(candidate, CLASS_NAMES):
            n = count_class_images(candidate, CLASS_NAMES)
            candidates.append((n, candidate))

    if candidates:
        candidates.sort(reverse=True)  # biggest first
        return candidates[0][1]

    return None

# ── Search known Kaggle mount points ─────────────────────────────────────────
DATA_DIR = None
for root_candidate in [
    Path('/kaggle/input/herlevdataset'),
    Path('/kaggle/input/datasets/shubhrawat132/herlevdataset'),
]:
    DATA_DIR = find_data_dir(root_candidate)
    if DATA_DIR:
        print(f'Found dataset at: {DATA_DIR}')
        break

if DATA_DIR is None:
    # Last resort: print everything under /kaggle/input to help debug
    print('ERROR: Could not find Herlev class folders!')
    print('Full /kaggle/input tree:')
    for p in sorted(Path('/kaggle/input').rglob('*')):
        if p.is_dir():
            print(f'  DIR  {p}')
        else:
            print(f'  FILE {p.parent}/{p.name}')
    raise FileNotFoundError(
        'Herlev dataset not found.\n'
        'Add it via: Add Data → shubhrawat132/herlevdataset'
    )

REPO_DIR    = Path('/kaggle/working/repo')
BACKEND_DIR = REPO_DIR / 'backend'
OUTPUT_DIR  = Path('/kaggle/working/checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Validate: make sure we have images, not just empty folders ────────────────
found_classes = [c for c in CLASS_NAMES if (DATA_DIR / c).exists()]
total_images  = count_class_images(DATA_DIR, CLASS_NAMES)

print(f'\nData dir      : {DATA_DIR}')
print(f'Classes found : {found_classes}')
print(f'Total images  : {total_images}')
print(f'Backend       : {BACKEND_DIR}')
print(f'Output        : {OUTPUT_DIR}')

if total_images < 10:
    raise ValueError(
        f'Only {total_images} images found in {DATA_DIR}.\n'
        'The folder may be empty or the dataset structure is unexpected.\n'
        'Run Cell 2 to inspect the full directory tree.'
    )

print('\nImage counts per class:')
for cls in CLASS_NAMES:
    cls_dir = DATA_DIR / cls
    if cls_dir.exists():
        count = len([f for f in cls_dir.iterdir() if f.is_file()])
        print(f'  {cls:<10} {count}')

# ── Install requirements ──────────────────────────────────────────────────────
req = BACKEND_DIR / 'requirements.txt'
if req.exists():
    print(f'\nInstalling requirements from {req} ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
    print('Done.')
else:
    print(f'\nNo requirements.txt at {req} — skipping pip install.')

for path in (str(REPO_DIR), str(BACKEND_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

# ── Suppress known harmless warnings ─────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore', message='.*var_limit.*')
warnings.filterwarnings('ignore', category=FutureWarning, message='.*GradScaler.*')
warnings.filterwarnings('ignore', category=FutureWarning, message='.*autocast.*')
warnings.filterwarnings('ignore', message='.*lr_scheduler.step.*')
print('\nWarning filters applied (FutureWarning/UserWarning suppressed).')


In [ ]:
# ── Cell 4 — Hyperparameters ──────────────────────────────────────────────────
# Edit values here directly. Environment variables override if set.

import os
from pathlib import Path

# Paths (set by Cell 3 — must run Cell 3 first)
OUTPUT_DIR  = Path('/kaggle/working/checkpoints')
DATA_DIR    = DATA_DIR   # resolved in Cell 3

# ─── Edit these ───────────────────────────────────────────────────────────────
EPOCHS             = int(os.environ.get('EPOCHS',             '60'))
BATCH_SIZE         = int(os.environ.get('BATCH_SIZE',         '16'))   # reduce to 8 if OOM
IMAGE_SIZE         = int(os.environ.get('IMAGE_SIZE',         '224'))
VAL_SPLIT          = float(os.environ.get('VAL_SPLIT',        '0.2'))
LR                 = float(os.environ.get('LR',               '5e-5'))
WEIGHT_DECAY       = float(os.environ.get('WEIGHT_DECAY',     '1e-4'))
PATIENCE           = int(os.environ.get('PATIENCE',           '15'))
ACCUMULATION_STEPS = int(os.environ.get('ACCUMULATION_STEPS', '2'))    # increase to 4 if OOM
WORKERS            = int(os.environ.get('WORKERS',            '2'))
SEED               = int(os.environ.get('SEED',               '42'))
GAMMA              = float(os.environ.get('GAMMA',            '2.0'))
BETA               = float(os.environ.get('BETA',             '0.9999'))
LABEL_SMOOTHING    = float(os.environ.get('LABEL_SMOOTHING',  '0.05'))
DROPOUT            = float(os.environ.get('DROPOUT',          '0.2'))
ACTIVATION         = os.environ.get('ACTIVATION',             'silu')
UNDERSAMPLE        = os.environ.get('UNDERSAMPLE',            'random')
LOSS_TYPE          = os.environ.get('LOSS_TYPE',              'class_balanced_focal')
USE_KFOLD          = os.environ.get('USE_KFOLD',              '0') == '1'
K_FOLDS            = int(os.environ.get('K_FOLDS',            '5'))
# ──────────────────────────────────────────────────────────────────────────────

print(f'Data dir            : {DATA_DIR}')
print(f'Output dir          : {OUTPUT_DIR}')
print(f'Epochs              : {EPOCHS}')
print(f'Batch size          : {BATCH_SIZE}')
print(f'Image size          : {IMAGE_SIZE}')
print(f'Learning rate       : {LR}')
print(f'Validation split    : {VAL_SPLIT}')
print(f'Loss type           : {LOSS_TYPE}')
print(f'Undersample         : {UNDERSAMPLE}')
print(f'Patience            : {PATIENCE}')
print(f'Accumulation steps  : {ACCUMULATION_STEPS}')
print(f'Use k-fold          : {USE_KFOLD}')

In [ ]:
# ── Cell 5 — Check train.py arguments ────────────────────────────────────────
# Prints the help text of train.py so we know exactly which flags it accepts.
# If the flag list differs from what Cell 6 passes, fix Cell 6 accordingly.

import subprocess, sys
from pathlib import Path

train_script = Path('/kaggle/working/repo/backend/train.py')
result = subprocess.run(
    [sys.executable, str(train_script), '--help'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

In [ ]:
# ── Cell 6 — Run training ─────────────────────────────────────────────────────
# Streams output in real-time. Best checkpoint is saved to OUTPUT_DIR after
# every epoch so a kernel timeout won't lose your work.
#
# NOTE: If Cell 5 shows that train.py does NOT accept some of the flags below
# (e.g. --backbone, --loss-type), remove those lines from the cmd list.

import os, subprocess, sys
from pathlib import Path

BACKEND_DIR  = Path('/kaggle/working/repo/backend')
train_script = BACKEND_DIR / 'train.py'

if not train_script.exists():
    raise FileNotFoundError(f'{train_script} not found — did Cell 1 run successfully?')

cmd = [
    sys.executable, str(train_script),
    '--data-dir',           str(DATA_DIR),
    '--output-dir',         str(OUTPUT_DIR),
    '--epochs',             str(EPOCHS),
    '--batch-size',         str(BATCH_SIZE),
    '--image-size',         str(IMAGE_SIZE),
    '--val-split',          str(VAL_SPLIT),
    '--lr',                 str(LR),
    '--weight-decay',       str(WEIGHT_DECAY),
    '--patience',           str(PATIENCE),
    '--workers',            str(WORKERS),
    '--seed',               str(SEED),
]

# ── Optional flags — train.py may or may not support these ───────────────────
# Cell 5 tells you which flags exist. Comment out any that aren't listed.
OPTIONAL_FLAGS = {
    '--accumulation-steps': str(ACCUMULATION_STEPS),
    '--loss-type':          LOSS_TYPE,
    '--undersample':        UNDERSAMPLE,
    '--gamma':              str(GAMMA),
    '--beta':               str(BETA),
    '--label-smoothing':    str(LABEL_SMOOTHING),
    '--activation':         ACTIVATION,
    '--dropout':            str(DROPOUT),
}

# Auto-detect which optional flags train.py accepts
help_result = subprocess.run(
    [sys.executable, str(train_script), '--help'],
    capture_output=True, text=True
)
help_text = help_result.stdout + help_result.stderr
for flag, value in OPTIONAL_FLAGS.items():
    if flag in help_text:
        cmd.extend([flag, value])
    else:
        print(f'Skipping unsupported flag: {flag}')

if USE_KFOLD and '--use-kfold' in help_text:
    cmd.extend(['--use-kfold', '--k-folds', str(K_FOLDS)])

print('Running:')
print(' '.join(cmd))
print('─' * 70)

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=env,
)
for line in proc.stdout:
    print(line, end='', flush=True)

exit_code = proc.wait()
if exit_code != 0:
    raise RuntimeError(f'Training failed with exit code {exit_code}')

print('\n' + '─' * 70)
print('Training complete!')

In [ ]:
# ── Cell 7 — Plot training curves & metrics ───────────────────────────────────
# Reads history.json and metrics.json written by train.py and produces:
#   • Loss curve (train vs val)
#   • Accuracy curve (train vs val)
#   • F1 curve (train vs val)
#   • Confusion matrix heatmap
#   • Per-class precision / recall / F1 bar chart

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

OUTPUT_DIR  = Path('/kaggle/working/checkpoints')
CLASS_NAMES = ['Normal', 'CIN1', 'CIN2', 'CIN3', 'Cancer']

# ── Load JSON files ───────────────────────────────────────────────────────────
history_path = OUTPUT_DIR / 'history.json'
metrics_path = OUTPUT_DIR / 'metrics.json'

if not history_path.exists():
    raise FileNotFoundError(f'{history_path} not found — run Cell 6 first.')

history = json.loads(history_path.read_text())
report  = json.loads(metrics_path.read_text()) if metrics_path.exists() else {}

epochs       = [e['epoch']        for e in history]
train_loss   = [e['train_loss']    for e in history]
val_loss     = [e['val_loss']      for e in history]
train_acc    = [e.get('train_accuracy', e.get('train_acc', 0)) for e in history]
val_acc      = [e.get('val_accuracy',   e.get('val_acc',   0)) for e in history]
train_f1     = [e.get('train_f1', 0)  for e in history]
val_f1       = [e.get('val_f1',   0)  for e in history]
val_auc      = [e.get('val_auc_roc', float('nan')) for e in history]

# ── Styling ───────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.size':        11,
})
BLUE  = '#2563EB'
RED   = '#DC2626'
GREEN = '#16A34A'
PURP  = '#7C3AED'

# ═══════════════════════════════════════════════════════════════════════════════
# Figure 1 — Training curves (2×2)
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training Curves — Cervical Cancer Classifier', fontsize=15, fontweight='bold', y=1.01)

# ── Loss ──────────────────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(epochs, train_loss, color=BLUE,  lw=2, label='Train Loss')
ax.plot(epochs, val_loss,   color=RED,   lw=2, label='Val Loss',  linestyle='--')
best_ep = epochs[int(np.argmin(val_loss))]
ax.axvline(best_ep, color='gray', lw=1, linestyle=':', label=f'Best epoch {best_ep}')
ax.set_title('Loss'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

# ── Accuracy ──────────────────────────────────────────────────────────────────
ax = axes[0, 1]
ax.plot(epochs, [v*100 for v in train_acc], color=BLUE, lw=2, label='Train Acc')
ax.plot(epochs, [v*100 for v in val_acc],   color=RED,  lw=2, label='Val Acc',  linestyle='--')
ax.set_title('Accuracy'); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 105); ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

# ── Macro F1 ──────────────────────────────────────────────────────────────────
ax = axes[1, 0]
ax.plot(epochs, train_f1, color=BLUE,  lw=2, label='Train F1')
ax.plot(epochs, val_f1,   color=GREEN, lw=2, label='Val F1',    linestyle='--')
ax.set_title('Macro F1'); ax.set_xlabel('Epoch'); ax.set_ylabel('F1 Score')
ax.set_ylim(0, 1.05); ax.legend()

# ── AUC-ROC ───────────────────────────────────────────────────────────────────
ax = axes[1, 1]
auc_valid = [(e, v) for e, v in zip(epochs, val_auc) if not (isinstance(v, float) and np.isnan(v))]
if auc_valid:
    ep_v, auc_v = zip(*auc_valid)
    ax.plot(ep_v, auc_v, color=PURP, lw=2, label='Val AUC-ROC')
    ax.set_ylim(0, 1.05)
else:
    ax.text(0.5, 0.5, 'AUC-ROC not available\n(single-class batches)', ha='center', va='center', transform=ax.transAxes, color='gray')
ax.set_title('AUC-ROC (macro OvR)'); ax.set_xlabel('Epoch'); ax.set_ylabel('AUC')
ax.legend()

plt.tight_layout()
curves_path = OUTPUT_DIR / 'training_curves.png'
plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {curves_path}')

# ═══════════════════════════════════════════════════════════════════════════════
# Figure 2 — Confusion matrix + per-class metrics
# ═══════════════════════════════════════════════════════════════════════════════
final = report.get('final_metrics', {})
cm    = final.get('confusion_matrix', [])
cr    = final.get('classification_report', {})

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Final Validation Metrics — Best Checkpoint', fontsize=14, fontweight='bold')

# ── Confusion matrix ──────────────────────────────────────────────────────────
ax = axes[0]
if cm:
    cm_arr = np.array(cm)
    present = [i for i, row in enumerate(cm_arr) if row.sum() > 0]
    labels  = [CLASS_NAMES[i] for i in present]
    cm_sub  = cm_arr[np.ix_(present, present)]

    # Normalise rows to % for readability
    cm_norm = cm_sub.astype(float)
    row_sums = cm_norm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    cm_pct = cm_norm / row_sums * 100

    im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100)
    plt.colorbar(im, ax=ax, label='% of true class')
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('Confusion Matrix (row-normalised %)')
    for i in range(len(labels)):
        for j in range(len(labels)):
            val = cm_sub[i, j]
            pct = cm_pct[i, j]
            color = 'white' if pct > 55 else 'black'
            ax.text(j, i, f'{int(val)}\n({pct:.0f}%)', ha='center', va='center', fontsize=9, color=color)
else:
    ax.text(0.5, 0.5, 'No confusion matrix data\n(run training first)', ha='center', va='center', transform=ax.transAxes, color='gray')
    ax.set_title('Confusion Matrix')

# ── Per-class precision / recall / F1 bar chart ───────────────────────────────
ax = axes[1]
if cr:
    plot_classes = [c for c in CLASS_NAMES if c in cr]
    prec  = [cr[c]['precision'] for c in plot_classes]
    rec   = [cr[c]['recall']    for c in plot_classes]
    f1s   = [cr[c]['f1-score']  for c in plot_classes]
    sup   = [int(cr[c]['support']) for c in plot_classes]

    x     = np.arange(len(plot_classes))
    width = 0.25
    bars_p = ax.bar(x - width, prec, width, label='Precision', color=BLUE,  alpha=0.85)
    bars_r = ax.bar(x,         rec,  width, label='Recall',    color=RED,   alpha=0.85)
    bars_f = ax.bar(x + width, f1s,  width, label='F1',        color=GREEN, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels([f'{c}\n(n={s})' for c, s in zip(plot_classes, sup)])
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score'); ax.set_title('Per-class Precision / Recall / F1')
    ax.legend()

    # Value labels on bars
    for bars in (bars_p, bars_r, bars_f):
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.2f}', ha='center', va='bottom', fontsize=8)

    # Macro averages as subtitle
    macro = cr.get('macro avg', {})
    wa    = cr.get('weighted avg', {})
    ax.set_xlabel(
        f"Macro  →  P: {macro.get('precision',0):.3f}  R: {macro.get('recall',0):.3f}  F1: {macro.get('f1-score',0):.3f}  "
        f"| Weighted  →  F1: {wa.get('f1-score',0):.3f}"
    )
else:
    ax.text(0.5, 0.5, 'No classification report data', ha='center', va='center', transform=ax.transAxes, color='gray')
    ax.set_title('Per-class Metrics')

plt.tight_layout()
metrics_fig_path = OUTPUT_DIR / 'metrics_chart.png'
plt.savefig(metrics_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {metrics_fig_path}')

# ═══════════════════════════════════════════════════════════════════════════════
# Summary table
# ═══════════════════════════════════════════════════════════════════════════════
print('\n' + '═'*60)
print('  FINAL BEST-CHECKPOINT METRICS (validation set)')
print('═'*60)
if final:
    print(f"  Accuracy          : {final.get('accuracy', 0)*100:.2f}%")
    print(f"  Macro Precision   : {final.get('precision', 0):.4f}")
    print(f"  Macro Recall      : {final.get('recall', 0):.4f}")
    print(f"  Macro F1          : {final.get('f1', 0):.4f}")
    auc = final.get('auc_roc', float('nan'))
    print(f"  AUC-ROC (macro)   : {auc:.4f}" if not (isinstance(auc, float) and (auc != auc)) else "  AUC-ROC (macro)   : N/A")
    print(f"  Val Loss          : {final.get('loss', 0):.4f}")
print('═'*60)
print(f'  Best epoch        : {best_ep}')
print(f'  Total epochs run  : {len(epochs)}')
print('═'*60)


In [ ]:
# ── Cell 8 — List saved artifacts ────────────────────────────────────────────

from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/checkpoints')

artifacts = sorted(OUTPUT_DIR.rglob('*.*'))
if not artifacts:
    print('No artifacts found in', OUTPUT_DIR)
    print('Training may not have saved checkpoints yet.')
else:
    print(f'Artifacts in {OUTPUT_DIR}:')
    total = 0
    for a in artifacts:
        size = a.stat().st_size / 1e6
        total += size
        print(f'  {a.name:<45} {size:>8.1f} MB')
    print(f'  {"TOTAL":<45} {total:>8.1f} MB')

In [ ]:
# ── Cell 9 — Zip for download ───────────────────────────────────────────────
# Download the zip from the Kaggle Output tab.
# Then in your local project root:
#   unzip herlev_checkpoints.zip
# Files land at: backend/Checkpoints/<filename>

import zipfile
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/checkpoints')
ZIP_PATH   = Path('/kaggle/working/herlev_checkpoints.zip')

CHECKPOINT_EXTS = {'.pt', '.pth', '.json', '.yaml', '.pkl', '.bin'}

artifacts = sorted(
    p for p in OUTPUT_DIR.rglob('*.*')
    if p.suffix in CHECKPOINT_EXTS
)

if not artifacts:
    raise FileNotFoundError(
        'No checkpoint files found. Make sure Cell 6 (training) completed successfully.'
    )

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for artifact in artifacts:
        arcname = str(Path('backend') / 'Checkpoints' / artifact.name)
        zf.write(artifact, arcname=arcname)
        print(f'  + {arcname}  ({artifact.stat().st_size/1e6:.1f} MB)')

print(f'\nZip size : {ZIP_PATH.stat().st_size / 1e6:.1f} MB')
print(f'Zip path : {ZIP_PATH}')
print()
print('Next steps:')
print('  1. Kaggle Output tab → Download herlev_checkpoints.zip')
print('  2. In your local project root:')
print('       unzip herlev_checkpoints.zip')
print('  3. Start the backend:')
print('       uvicorn backend.api.main:app --reload')

## Troubleshooting

| Problem | Cause | Fix |
|---|---|---|
| `git clone` fails | Internet is OFF | Notebook Settings → Internet → ON |
| `NameError: DATA_DIR` | Cells run out of order | Run All (Run → Run All Cells) |
| `FileNotFoundError: herlevdataset` | Dataset not attached | Add Data → `shubhrawat132/herlevdataset` |
| CUDA out of memory | Batch too large | Set `BATCH_SIZE = 8`, `ACCUMULATION_STEPS = 4` in Cell 4 |
| Flag not recognized error | train.py doesn't support that arg | Cell 5 shows which flags exist; Cell 6 auto-skips unsupported ones |
| Kernel timeout mid-run | Kaggle 9-hour limit | Best checkpoint already saved; run Cell 7 + 8 to zip it |